In [2]:
## loading packages
import pandas as pd
import numpy as np
import plotly.graph_objects as go
#import plotly.express as px
import argparse
import sys
import re
import random
import difflib
from collections import defaultdict

import logging

# Configure once (usually at program start)
logging.basicConfig(
    level=logging.INFO,  # minimum level to display
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logging.getLogger().setLevel(logging.DEBUG)

def is_notebook():
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            return True   # Jupyter notebook or qtconsole
        elif shell == 'TerminalInteractiveShell':
            return False  # Terminal running IPython
        else:
            return False  # Other type (?)
    except Exception:
        return False      # Probably standard Python interpreter

In [3]:
## argparse definition
if not is_notebook():
    script_name = sys.argv[0]
else:
    script_name = 'Script'

parser = argparse.ArgumentParser(description=f"{script_name} parameters")

## eigenvectors
parser.add_argument('--eigenvec', type=str, default=None, required=False, help='Eigenvec file path (plink)')
parser.add_argument('--eigenvecID', type=str, default=None, help='Eigenvec ID column (default first column)')

## eigenvalues
parser.add_argument('--eigenval', type=str, default=None, required=False, help='Eigenval file path (plink)')
#parser.add_argument('--reduction', type=str, default='MDS', help='Reduction method')
parser.add_argument('--nb_eigenvalues', type=int, default=10, help='Number of eigenvalues to show (0 for all)')

## stats
parser.add_argument('--imiss', type=str, default=None, help='Imiss file path (plink)')
parser.add_argument('--lmiss', type=str, default=None, help='Lmiss file path (plink)')
parser.add_argument('--frq', type=str, default=None, help='Frq file path (plink)')

## annotation
parser.add_argument('--annotation', type=str, default=None, help='Annotation file path')
parser.add_argument('--annotationID', type=str, default='Genetic ID', help='Annotation ID column (default first column)')
parser.add_argument('--longitude', type=str, default=None, help='Longitude column name')
parser.add_argument('--latitude', type=str, default=None, help='Latitude column name')
parser.add_argument('--time', type=str, default=None, help='Time column name')
#parser.add_argument('--group', type=str, default=None, help='Grouping column name')

## text handling
parser.add_argument('--ignore_case', action='store_true', default=None, help='Ignore case differences in annotation')
parser.add_argument('--no_ignore_case', dest='ignore_case', default=None, action='store_true', help='Do not ignore case differences in annotation')
parser.set_defaults(ignore_case=True)

parser.add_argument('--ignore_space', action='store_true', default=None, help='Ignore space differences in annotation')
parser.add_argument('--no_ignore_space', dest='ignore_space', default=None, action='store_true', help='Do not ignore space differences in annotation')
parser.set_defaults(ignore_space=True)

parser.add_argument('--col_abbrev', type=int, default=15, help='Abbreviate column names to this length (0 for no abbreviation)')
parser.add_argument('--legend_abbrev', type=int, default=0, help='Abbreviate legend text to this length (0 for no abbreviation)')

parser.add_argument('--max_factors', type=int, default=400, help='Maximum number of different elements in factorial columns to be included')

## time figure
parser.add_argument('--time_hist', action='store_true', default=None, help='Show points in time as histogram')
parser.add_argument('--time_scatter', dest='time_hist', action='store_true', default=None, help='Show points in time as scatter plot')
parser.set_defaults(time_hist=True)
parser.add_argument('--time_hist_nbins', type=int, default=100, help='Number of bins for the tme histogram (100)')

## plot settings
parser.add_argument('--hover_minimal', action='store_true', default=None, help='Show minimal information when hovering points')
parser.add_argument('--hover_full', dest='hover_minimal', action='store_false', default=None, help='Show full information when hovering points')
parser.set_defaults(hover_minimal=True)

## server settings
parser.add_argument('--use_server', action='store_true', default=None, help='Use a dash server for interactive plots')
parser.add_argument('--no_server', dest='use_server', default=None, action='store_false', help='Do not use a dash server: less interactive but static HTML output')
parser.set_defaults(use_server=True)

parser.add_argument('--open_browser', action='store_true', default=None, help='Open directly the dash server in a web browser')
parser.add_argument('--no_open_browser', dest='open_browser', default=None, action='store_false', help='Do not open the dash server in a web browser: you will have to open it manually')
parser.set_defaults(open_browser=False)

parser.add_argument('--server_port', type=int, default=8050, help='Port for the dash server (default 8050)')

## self-contained HTML
parser.add_argument('--html', action='store_true', default=None, help='Generate self-contained HTML')
parser.add_argument('--no_html', dest='html', action='store_true', default=None, help='Do not generate self-contained HTML')
parser.set_defaults(html=False)
parser.add_argument('--html_file', type=str, default=None, help='File name for self-contained HTML (default: eigenvec file name + .html)')


## development
parser.add_argument('--dev', action='store_true', default=None, help='Use development parameters')
parser.add_argument('--no_dev', dest='dev', action='store_false', default=None, help='Do not use development parameters')
parser.set_defaults(dev=False)

#parser.print_help()

In [4]:

## development arguments
# Parse NOTHING when running in a notebook
if is_notebook():
    args = parser.parse_args([])      # <— key line ([])
else:
    args = parser.parse_args()

if is_notebook() or args.dev:
    args.eigenvec='aadr.eigenvec'
    args.eigenval='aadr.eigenval'

    args.imiss='aadr.imiss'
    args.lmiss='aadr.lmiss'
    args.frq='1000gp.frq'

    args.annotation='aadr.anno'
    
    args.reduction='MDS'
    args.eigenvecID='sample'
    args.annotationID='Genetic ID'
    args.nb_eigenvalues=10
    args.longitude='Long.'
    args.latitude='Lat.'
    args.time='zDate mean in BP in years before 1950 CE [OxCal mu for a direct radiocarbon date, and average of range for a contextual date]'
    args.group='Political_Entit'

    args.ignore_case=True
    args.ignore_space=True

    args.col_abbrev=15
    args.legend_abbrev=20

    args.max_factors=400

    args.use_server=False
    args.open_browser=False
    args.server_port=8050
    args.time_hist_nbins=500



if not args.dev and not args.eigenvec:
    logging.error('Missing argument: --eigenvec  is required.')

In [5]:
## functions

## abbreviate a list of strings to a maximum length, preserving uniqueness
def make_unique_abbr(cur_list, max_length=3):
    # Compile regex patterns once
    clean_re = re.compile(r'[^a-zA-Z0-9 ]')
    space_re = re.compile(r' ')

    # Clean and abbreviate
    cleaned = [clean_re.sub('', elem) for elem in cur_list]
    abbreviated = [space_re.sub('_', elem[:max_length]).rstrip('_') for elem in cleaned]

    # Ensure uniqueness using a counter
    counter = defaultdict(int)
    unique_abbr = []

    for abbr in abbreviated:
        new_abbr = abbr
        while new_abbr in counter:
            counter[abbr] += 1
            new_abbr = f"{abbr}{counter[abbr]}"
        counter[new_abbr] = 0
        unique_abbr.append(new_abbr)

    return unique_abbr


## abbreviate columns of a pandas DataFrame
def make_unique_abbr_of_df(df, cols, max_length=3):
    """
    Abbreviate the values in the specified columns of a DataFrame.
    Each unique value in the column is replaced by a unique abbreviation.
    """
    for col in cols:
        unique_vals = df[col].astype(str).unique()
        abbrs = make_unique_abbr(unique_vals, max_length)
        abbr_map = dict(zip(unique_vals, abbrs))
        df[col] = df[col].astype(str).map(abbr_map)
    return df


## search the the best text match within a list of words
def get_abbr_of(target, lookup, returns=None):
    closest = difflib.get_close_matches(target, lookup, n=1)
    if closest is None or len(closest) == 0:
        print(f"Warning: No match found for {target} in {lookup}")
        return None
    
    if returns is  None:
        match = closest[0]
    else:
        match = returns[lookup.index(closest[0])]

    #print(f"Look up for {target} => {match}")
    return match



## de-duplex columns ignoring capitalization and space differences (keep first version of each)
def deduplicate_columns(df, columns, ignore_case=True, ignore_space=True):
    for col in columns:
        norm_col = f'_norm_{col}'
        series = df[col].astype(str)
        if ignore_case:
            series = series.str.lower()
        if ignore_space:
            series = series.str.replace(r'\s+', '', regex=True)
        df[norm_col] = series
        first_map = df.drop_duplicates(norm_col, keep='first').set_index(norm_col)[col]
        df[col] = df[norm_col].map(first_map)
        df.drop(columns=[norm_col], inplace=True)
    return df

## find automatically the pca dimensions prefix
def find_incrementing_prefix_series(columns):
    # Match prefix + number, e.g., PC1, PC2, Dim3
    pattern = re.compile(r'^([A-Za-z_]+)(\d+)$')
    prefix_groups = defaultdict(list)

    # Group columns by prefix
    for col in columns:
        match = pattern.match(col)
        if match:
            prefix, num = match.groups()
            prefix_groups[prefix].append(int(num))

    # Find prefixes with longest incrementing series
    longest_series = []
    for prefix, nums in prefix_groups.items():
        nums_sorted = sorted(nums)
        # Check if numbers form a consecutive sequence
        if nums_sorted == list(range(nums_sorted[0], nums_sorted[-1] + 1)):
            series = [f"{prefix}{n}" for n in nums_sorted]
            if len(series) > len(longest_series):
                longest_series = series

    return longest_series

In [6]:
## reading imiss
if args.imiss is not None:
    logging.info(f"Reading imiss file '{args.imiss}' ...")
    df_imiss = pd.read_csv(args.imiss, sep=r"\s+", skiprows=[1])
    logging.info(f"Reading imiss file '{args.imiss}' ... done.")
else:
    df_imiss = None

2025-09-13 22:13:05,179 [INFO] Reading imiss file 'aadr.imiss' ...
2025-09-13 22:13:05,236 [INFO] Reading imiss file 'aadr.imiss' ... done.


In [7]:
## reading lmiss
if args.lmiss is not None:
    logging.info(f"Reading lmiss file '{args.lmiss}' ...")
    df_lmiss = pd.read_csv(args.lmiss, sep=r"\s+", skiprows=[1], low_memory=False)
    logging.info(f"Reading lmiss file '{args.lmiss}' ... done.")
else:
    df_lmiss = None

2025-09-13 22:13:05,241 [INFO] Reading lmiss file 'aadr.lmiss' ...
2025-09-13 22:13:06,555 [INFO] Reading lmiss file 'aadr.lmiss' ... done.


In [8]:
## reading frq
if args.frq is not None:
    logging.info(f"Reading frq file '{args.frq}' ...")
    df_frq = pd.read_csv(args.frq, sep=r"\s+")
    logging.info(f"Reading frq file '{args.frq}' ... done.")
else:
    df_frq = None


2025-09-13 22:13:06,560 [INFO] Reading frq file '1000gp.frq' ...
2025-09-13 22:13:06,568 [INFO] Reading frq file '1000gp.frq' ... done.


In [9]:
## reading eigenval

if args.eigenval is not None:
    logging.info(f"Reading eigenval file '{args.eigenval}' ...")
    eigenval = pd.read_csv(args.eigenval, sep="\t", header=None, names=["eigenvalue"])

    logging.info(f"   Found {len(eigenval)} eigenvalues.")

    ## compute variance explained
    eigenval["eigenvalue"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum()

    ## add cumulative eigenvalues
    eigenval["cumulative"] = eigenval["eigenvalue"].cumsum()

    ## add index starting from 1 in the first column
    eigenval["dimension"] = eigenval.index + 1

    ## order columns
    eigenval = eigenval[["dimension"] + [col for col in eigenval.columns if col != "dimension"]]
else:
    eigenval = None

logging.info(f"Reading eigenval file '{args.eigenval}' ... done.")

2025-09-13 22:13:06,573 [INFO] Reading eigenval file 'aadr.eigenval' ...
2025-09-13 22:13:06,576 [INFO]    Found 10442 eigenvalues.
2025-09-13 22:13:06,581 [INFO] Reading eigenval file 'aadr.eigenval' ... done.


In [10]:
## reading eigenvec (always present)

# Read eigenvectors
logging.info(f"Reading eigenvec file '{args.eigenvec}' ...")
eigenvec = pd.read_csv(args.eigenvec, sep=r"\s+", header=0)

PCS = find_incrementing_prefix_series(eigenvec.columns)

if len(PCS) < 2:
    logging.error('Not enough dimension columns found in the eigenvec file ({len(PCS)} found)).')

logging.info(f"   Found {len(PCS)} principal components ({", ".join(PCS[:2])}, ...).")

# Scale PC columns: to be removed
nb_snp = 194926 # Number of SNPs
eigenvec[PCS] = eigenvec[PCS].div(nb_snp)

logging.info(f"Reading eigenvec file '{args.eigenvec}' ... done.")


2025-09-13 22:13:06,585 [INFO] Reading eigenvec file 'aadr.eigenvec' ...
2025-09-13 22:13:06,613 [INFO]    Found 20 principal components (C1, C2, ...).
2025-09-13 22:13:06,616 [INFO] Reading eigenvec file 'aadr.eigenvec' ... done.


In [11]:
## reading annotation
annotation = None
ANNOTATION_TIME = None
ANNOTATION_LAT = None
ANNOTATION_LONG = None

if args.annotation is not None:
    logging.info(f"Reading annotation file '{args.annotation}' ...")
    annotation = pd.read_csv(args.annotation, sep='\t', na_values='..', low_memory=False)
    
    # create abbreviation_desc data.frame
    annotation_desc = pd.DataFrame({
        'Abbreviation': make_unique_abbr(annotation.columns, max_length=args.col_abbrev),
        'Description': annotation.columns,
        'Type': [annotation[col].dtype.name for col in annotation.columns]
    })

    # Add a column for number of unique elements if the column is categorical
    n_unique = []
    for col in annotation.columns:
        dtype = annotation[col].dtype.name
        if dtype in ['object', 'category']:
            n_unique.append(annotation[col].nunique(dropna=False))
        else:
            n_unique.append(None)
    annotation_desc['N_levels'] = n_unique
    
    # Rename columns of annotation data.frame with the abbreviations
    annotation.columns = annotation_desc['Abbreviation']

    # Get abbreviations for given IDs
    if args.annotationID is not None:
        ANNOTATION_ID = get_abbr_of(args.annotationID, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_ID not in annotation.columns:
            logging.info(f"Warning: Specified annotationID '{--annotationID}' not found in annotation columns. Using first column instead.")
    else:
        ANNOTATION_ID = annotation.columns[0] ## default is the first column

    if args.time is not None:
        ANNOTATION_TIME = get_abbr_of(args.time, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_TIME not in annotation.columns:
            ANNOTATION_TIME = None
            logging.warning(f"Warning: Specified time '{--time}' not found in annotation columns. Time graph disabled.")

    if args.longitude is not None:
        ANNOTATION_LONG = get_abbr_of(args.longitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LONG not in annotation.columns:
            ANNOTATION_LONG = None
            logging.warning(f"Warning: Specified longitude '{--longitude}' not found in annotation columns. Geographical map disabled.")

    if args.latitude is not None:
        ANNOTATION_LAT = get_abbr_of(args.latitude, annotation_desc['Description'].to_list(), annotation_desc['Abbreviation'].to_list())
        if ANNOTATION_LAT not in annotation.columns:
            ANNOTATION_LAT = None
            logging.warning(f"Warning: Specified latitude '{--latitude}' not found in annotation columns. Geographical map disabled.")

    ## clean factorial elements if needed
    exclude_abbr = [elem for elem in [ANNOTATION_ID, ANNOTATION_TIME, ANNOTATION_LONG, ANNOTATION_LAT] if elem != None]

    FACTORIAL_COLUMNS = annotation_desc.loc[
        (annotation_desc['N_levels'].notnull()) &
        (annotation_desc['N_levels'] <= args.max_factors) &
        (~annotation_desc['Abbreviation'].isin(exclude_abbr)),
        'Abbreviation'
    ].tolist()
    
    CONTINUOUS_COLUMNS = [col for col in annotation.columns if col not in FACTORIAL_COLUMNS]
    
    if args.ignore_case or args.ignore_space:
        if args.ignore_case:
            logging.info(f"   Ignoring case differences.")
        if args.ignore_space:
            logging.info(f"   Ignoring space differences.")
        annotation = deduplicate_columns(annotation, FACTORIAL_COLUMNS, args.ignore_case, args.ignore_space)

    if args.legend_abbrev > 0:
        logging.info(f"   Shortening legend text to max {args.legend_abbrev} characters.")
        annotation = make_unique_abbr_of_df(annotation, FACTORIAL_COLUMNS, args.legend_abbrev)

    ## logging
    logging.info(f"   Found {len(FACTORIAL_COLUMNS)} factorial columns.")
    logging.info(f"   Found {len(CONTINUOUS_COLUMNS)} continuous columns.")
    if ANNOTATION_LAT and ANNOTATION_LONG:
        logging.info(f"   Found geographic coordinates columns (latitude/longitude): '{ANNOTATION_LAT}', '{ANNOTATION_LONG}'.")
    if ANNOTATION_TIME:
        logging.info(f"   Found time column: '{ANNOTATION_TIME}'.")
    
    logging.info(f"Reading annotation file '{args.annotation}' ... done.")

2025-09-13 22:13:06,625 [INFO] Reading annotation file 'aadr.anno' ...
2025-09-13 22:13:06,743 [INFO]    Ignoring case differences.
2025-09-13 22:13:06,743 [INFO]    Ignoring space differences.
2025-09-13 22:13:06,893 [INFO]    Shortening legend text to max 20 characters.
2025-09-13 22:13:06,917 [INFO]    Found 10 factorial columns.
2025-09-13 22:13:06,917 [INFO]    Found 32 continuous columns.
2025-09-13 22:13:06,918 [INFO]    Found geographic coordinates columns (latitude/longitude): 'Lat', 'Long'.
2025-09-13 22:13:06,918 [INFO]    Found time column: 'Date_mean_in_BP'.
2025-09-13 22:13:06,918 [INFO] Reading annotation file 'aadr.anno' ... done.


In [12]:
## merging coord and annotation

# Check if annotation exists (not None and not empty)
if annotation is not None and not annotation.empty:
    EIGENVEC_ID = get_abbr_of(args.eigenvecID, eigenvec.columns.to_list()) if args.eigenvecID is not None else eigenvec.columns[0] ## default is the first column

    # Rename EIGENVEC_ID column in annotation to match EIGENVEC_ID in eigenvec
    # so we can join by that column name
    annotation_renamed = annotation.rename(columns={ANNOTATION_ID: EIGENVEC_ID})
    
    # Perform left join
    coord = eigenvec.merge(annotation_renamed, on=EIGENVEC_ID, how="left")
    
    # Negate the values in the ANNOTATION_TIME column
    coord[ANNOTATION_TIME] = -coord[ANNOTATION_TIME]
else:
    coord = eigenvec

# Rename the EIGENVEC_ID column to id
coord = coord.rename(columns={EIGENVEC_ID: "id"})

# Move 'id' column to the front
cols = ['id'] + [c for c in coord.columns if c != 'id']
coord = coord[cols]


In [13]:
## init variables

init_x = PCS[0]
init_y = PCS[1]

sizes = [4, 8, 12, 16]
init_size = 8

In [ ]:
import dash
from dash import dcc, html, Input, Output, State, ctx
from dash_ag_grid import AgGrid
import plotly.express as px
import pandas as pd
from functools import lru_cache
from dash import callback_context
import time
from functools import wraps


# Replace these with your actual variables
df = coord  # Assumed to be preloaded

def get_annotation_table():
    # Get the column names of coord (PCS at the end)
    all_columns = [col for col in coord if col not in PCS] + PCS

    # Use a dict for fast lookup instead of merge
    abbr_map = {abbr: desc for abbr, desc in zip(annotation_desc['Abbreviation'], annotation_desc['Description'])}
    type_map = {abbr: dtype for abbr, dtype in zip(annotation_desc['Abbreviation'], annotation_desc['Type'])}
    nlevels_map = {abbr: nlev for abbr, nlev in zip(annotation_desc['Abbreviation'], annotation_desc['N_levels'])}

    # Build the table rows directly
    rows = []
    for abbr in all_columns:
        rows.append({
            'Abbreviation': abbr,
            'Description': abbr_map.get(abbr, ''),
            'Type': type_map.get(abbr, ''),
            'N_levels': nlevels_map.get(abbr, '')
        })
    return pd.DataFrame(rows)

extended_annotation_table = get_annotation_table()

##-----------------------------------------------------------------------------
## Annotation tab
def get_annotation_tab():
    logging.debug(f"get_annotation_tab() ... ")
    if annotation is None:
        return None

    tab = html.Div(
        id='annotation_tab_content',
        style={'height': '100%', 'width': '100%', 'display': 'flex', 'flexDirection': 'column'},
        children=[
            html.Div([
                AgGrid(
                    id='annotation-table',
                    rowData=extended_annotation_table.to_dict('records'),
                    columnDefs=[
                        {'headerName': '', 'checkboxSelection': True, 'headerCheckboxSelection': True, 'width': 40},
                        *[
                            {
                                'headerName': col,
                                'field': col,
                                'sortable': True,
                            }
                            for col in extended_annotation_table.columns
                        ]
                    ],
                    dashGridOptions={
                        "rowSelection": "multiple",
                        "rowMultiSelectWithClick": True,
                        "pagination": True,
                        "paginationAutoPageSize": 100,
                        "defaultColDef": {
                            "resizable": True,
                            "wrapText": True,
                            "autoHeight": True,
                        },
                    },
                    className="ag-theme-alpine",
                    selectedRows=extended_annotation_table.to_dict('records')[:10],
                    style={'height': '100%', 'width': '100%'}
                )
            ], style={'flex': 1, 'height': '100%', 'width': '100%'})
        ]
    )
    logging.debug(f"get_annotation_tab() ... done.")
    return tab

##-----------------------------------------------------------------------------
## Eigenvalues tab
def get_eigenvalues_tab():
    logging.debug(f"get_eigenvalues_tab() ... ")
    if args.eigenval is None:
        return None

    tab = html.Div([
        html.Div([
            dcc.Graph(
                id='eigenvals',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['eigenvalue'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Eigenvalues'
                    }],
                    'layout': {
                        'title': {'text': 'Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': 'calc(100vh - 50px - 50px)', 'display': 'inline-block', 'verticalAlign': 'top'}
            ),
            dcc.Graph(
                id='eigenvals_cumulative',
                figure={
                    'data': [{
                        'x': eigenval["dimension"][:args.nb_eigenvalues],
                        'y': 100 * eigenval['cumulative'][:args.nb_eigenvalues],
                        'type': 'bar',
                        'name': 'Cumulative'
                    }],
                    'layout': {
                        'title': {'text': 'Cumulative Eigenvalues'},
                        'xaxis': {'title': {'text': 'Dimension'}},
                        'yaxis': {'title': {'text': '% cumulative explained variance'}},
                        'autosize': True,
                        'height': None,
                    }
                },
                style={'width': '50%', 'height': 'calc(100vh - 50px - 50px)', 'display': 'inline-block', 'verticalAlign': 'top'}
            ),
        ], style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], id='eigenvalues_tab_content', style={'height': '100vh', 'width': '100%'})

    logging.debug(f"get_eigenvalues_tab() ... done")
    return tab


##-----------------------------------------------------------------------------
## Statistics tab
def get_statistics_tab():
    logging.debug(f"get_statistics_tab() ... ")
    if df_imiss is None and df_lmiss is None and df_frq is None:
        return None  # Don't render the tab at all

    graphs = []

    ## imiss
    if df_imiss is not None:
        fig_imiss = px.histogram(
            df_imiss, x='F_MISS', nbins=50, title='Sample missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_imiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='imiss', figure=fig_imiss, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    ## lmiss
    if df_lmiss is not None:
        fig_lmiss = px.histogram(
            df_lmiss, x='F_MISS', nbins=50, title='SNP missing rate',
            labels={'F_MISS': 'Missing rate', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_lmiss.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 1]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='lmiss', figure=fig_lmiss, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    ## freq
    if df_frq is not None:
        fig_frq = px.histogram(
            df_frq, x='MAF', nbins=500, title='Minor Allele Frequency',
            labels={'MAF': 'Allele frequency', 'count': 'Count'},
            color_discrete_sequence=['#1F77B4']
        )
        fig_frq.update_layout(
            template='plotly_white',
            xaxis=dict(range=[0, 0.5]), 
            autosize=True
        )
        graphs.append(
            dcc.Graph(id='freq', figure=fig_frq, style={
                'flex': 1, 'height': 'calc(100vh - 50px - 50px)', 'width': '100%',
                'display': 'inline-block', 'verticalAlign': 'top'
            })
        )

    tab = html.Div([
        html.Div(graphs, style={
            'width': '100%',
            'height': '100%',
            'display': 'flex',
            'flexDirection': 'row',
            'justifyContent': 'space-between',
            'alignItems': 'stretch'
        }),
    ], id='statistics_tab_content', style={'height': '100vh', 'width': '100%'})

    logging.debug(f"get_statistics_tab() ... done.")
    return tab


##-----------------------------------------------------------------------------
## Help tab
def get_help_tab():
    logging.debug(f"get_help_tab() ... ")
    tab = html.Div([
        ## Hover style
        html.Div("Hover style:", style={"marginRight": "8px"}),
        dcc.RadioItems(
            id='hover-switch',
            options=[
                {'label': 'Minimal', 'value': True},
                {'label': 'Full', 'value': False}
            ],
            value=args.hover_minimal,
            inline=True,
            style={'marginLeft': '5px'}
        ),


        ## time plot
        html.Div("Time plot:", style={"marginRight": "8px"}),
        dcc.RadioItems(
            id='time_plot-switch',
            options=[
                {'label': 'Histogram', 'value': True},
                {'label': 'Scatterplot', 'value': False}
            ],
            value=args.time_hist,
            inline=True,
            style={'marginLeft': '5px'}
        ),
        
        #argparse
        html.Pre(parser.format_help())
    ], id = 'help_tab_content')
    logging.debug(f"get_help_tab() ... done.")
    return tab


##-----------------------------------------------------------------------------
## PCA tab

def get_selectors():
    logging.debug(f"   get_selectors() ... ")
    dropdowns = []

    # X-axis
    dropdowns.append(
        html.Div([
            html.Div("X-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='x-axis-selector', 
                         options=PCS, 
                         value=init_x,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Y-axis
    dropdowns.append(
        html.Div([
            html.Div("Y-axis:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='y-axis-selector', 
                         options=PCS, 
                         value=init_y,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
    )

    # Grouping (optional)
    if FACTORIAL_COLUMNS is not None:
        dropdowns.append(
            html.Div([
                html.Div("Grouping:", style={"marginRight": "5px"}),
                dcc.Dropdown(id='group-selector', 
                             options=FACTORIAL_COLUMNS,
                             value=FACTORIAL_COLUMNS[0],
                             clearable=False, 
                             style={'width': '200px'}),
            ], style={'display': 'flex', 'alignItems': 'center', 'marginRight': '20px'})
        )

    # Point size
    dropdowns.append(
        html.Div([
            html.Div("Point size:", style={"marginRight": "5px"}),
            dcc.Dropdown(id='point-size-selector', 
                         options=sizes, 
                         value=8,
                         clearable=False, 
                         style={'width': '120px'}),
        ], style={'display': 'flex', 'alignItems': 'center'})
    )
    selectors = html.Div(dropdowns, style={'display': 'flex', 'flexWrap': 'wrap', 'marginBottom': '15px'})
    logging.debug(f"   get_selectors() ... done.")
    return selectors

def get_data_table():
    logging.debug(f"   get_data_table() ... ")
    selected_columns = extended_annotation_table['Abbreviation'].head(10).tolist()
    defs = [{'headerName': col, 'field': col, 'sortable': True, 'resizable': True, 
             **({'checkboxSelection': True, 'headerCheckboxSelection': True} if col == 'id' else {})} 
             for col in selected_columns]

    tab = html.Div(
        AgGrid(
            id='data-table',
            rowData=df.to_dict('records'),
            columnDefs=defs,
            selectedRows=df.to_dict('records'),
            dashGridOptions={
                "rowSelection": "multiple",
                "pagination": True,
                "paginationAutoPageSize": 100
            },
            className="ag-theme-alpine",
            style={'height': '100%', 'width': '100%', 'overflowX': 'auto'}
        ),
        style={'height': '50%', 'width': '100%'}
    )
    logging.debug(f"   get_data_table() ... done.")
    return tab

def get_pca_tab():
    logging.debug(f"get_pca_tab() ... ")
    # LEFT column children (always present)
    left_children = [
        get_selectors(),
        dcc.Graph(id='pca-plot', style={'height': 'calc(100% - 10cm)', 'width': '100%'})
    ]
    if ANNOTATION_TIME is not None:
        left_children.append(
            dcc.Graph(id='time-plot', style={'height': '10cm', 'width': '100%'})
        )

    # RIGHT column children (optional)
    right_children = []
    if ANNOTATION_LAT is not None and ANNOTATION_LONG is not None:
        ## map plot
        right_children.append(
            html.Div(
                dcc.Graph(id='map-plot', style={'height': '100%', 'width': '100%'}),
                style={'height': '50%', 'width': '100%'}
            )
        )

    if annotation is not None:
        ## data table
        right_children.append(get_data_table())

        ## filter
        right_children.append(
            html.Div([
                dcc.Textarea(id='texarea-expression', style={'width': '100%', 'height': '80px'}),
            ])
        )

    # Columns: LEFT always present, RIGHT optional
    columns_children = [
        html.Div(left_children,
                 style={'flex': 1, 'padding': '10px', 'minWidth': 0,
                        'display': 'flex', 'flexDirection': 'column'})
    ]
    if right_children:
        columns_children.append(
            html.Div(right_children,
                     style={'flex': 1, 'padding': '10px', 'minWidth': 0,
                            'display': 'flex', 'flexDirection': 'column'})
        )

    # Main tab layout
    tab = html.Div(
        id='pca_tab_content',
        children=[
            html.Div(
                id='selection-count',
                style={'padding': '10px', 'fontWeight': 'bold'},
                children=f"Showing all {len(df)} points"
            ),
            html.Div(columns_children,
                     style={'display': 'flex', 'width': '100%', 'height': 'calc(100vh - 50px - 100px)'}
            )
        ]
    )
    logging.debug(f"get_pca_tab() ... done.")
    return tab






###---------------------------------------------------
## get tabs dynamically
def labeled_tabs(visible_tab_ids):
    tab_labels = {
        'pca_tab': 'PCA',
        'annotation_tab': 'Annotation',
        'eigenvalues_tab': 'Eigenvalues',
        'statistics_tab': 'Statistics',
        'help_tab': 'Help'
    }

    return html.Div(
        style={'display': 'flex', 'alignItems': 'center', 'width': '100%'},
        children=[
            # Left title
            html.Div(
                "interactivePCA",
                style={
                    'fontWeight': 'bold',
                    'marginRight': '20px',
                    'fontSize': '20px',
                    'whiteSpace': 'nowrap'
                }
            ),

            # Tabs
            html.Div(
                style={'flex': 1},  # make tabs container take remaining space
                children=dcc.Tabs(
                    id='tabs',
                    value=visible_tab_ids[0] if visible_tab_ids else None,
                    style={'width': '100%'},  # stretch container
                    children=[
                        dcc.Tab(
                            label=tab_labels[tab_id],
                            value=tab_id,
                            style={
                                'flex': 1,              # each tab stretches equally
                                'textAlign': 'center'
                            },
                            selected_style={
                                'flex': 1,
                                'textAlign': 'center',
                                'fontWeight': 'bold'
                            }
                        )
                        for tab_id in visible_tab_ids
                    ]
                )
            )
        ]
    )


# Build tabs dynamically (Filter out tabs where the *content* is None)
tab_contents = [
    ('pca_tab', get_pca_tab()),
    ('annotation_tab', get_annotation_tab()),
    ('eigenvalues_tab', get_eigenvalues_tab()),
    ('statistics_tab', get_statistics_tab()),
    ('help_tab', get_help_tab())
]
tab_contents = [(tab_id, content) for tab_id, content in tab_contents if content is not None]

# Extract IDs and contents
tab_ids = [tab_id for tab_id, _ in tab_contents]
tab_components = [content for _, content in tab_contents]

###---------------------------------------------------
###---------------------------------------------------
## main layout
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Store(id='selected-ids', data=df['id'].tolist()), # Store all selected ids
    dcc.Store(id="visible-groups", data={}), # Store visible groups for coloring
     dcc.Store(id='color-map', data={}), # Store color map for groups
    dcc.Store(id='visible-columns', data=extended_annotation_table['Abbreviation'].head(10).tolist()), # Store visible columns in the data table
    labeled_tabs(tab_ids),
    html.Div(id='tabs-content', children=tab_components)
])


###---------------------------------------------------
###---------------------------------------------------
## Callback

# Callback to switch tab content
@app.callback(
    [Output(f"{tab_id}_content", "style") for tab_id in tab_ids],
    Input("tabs", "value")
)
def display_tab_content(active_tab):
    return [
        {'display': 'block'} if tab_id == active_tab else {'display': 'none'}
        for tab_id in tab_ids
    ]


## -------------------------------------------------------------------
###  shared figure helpers
def get_selected_df(selected_ids):
    return df[df['id'].isin(selected_ids)]

def get_unselected_df(selected_ids):
    return df[~df['id'].isin(selected_ids)] 

## hover text
def get_hover_text(columns):
    lines = [f"<b>{col}</b>: %{{customdata[{i}]}}" for i, col in enumerate(columns)]

    # insert an extra empty line after the first element
    if len(lines) >= 1:
        lines.insert(1, "")

    return "<br>".join(lines) + "<extra></extra>"


###---------------------------------------------------
## PCA
def get_pca_plot(selected_ids, x_col, y_col, group, point_size, color_map, visible_columns, minimal_hover=False):
    selected_df = get_selected_df(selected_ids)
    unselected_df = get_unselected_df(selected_ids)

    fig = go.Figure()

    # Unselected points (gray)
    if len(unselected_df) > 0:
        fig.add_trace(
            go.Scattergl(
                x=unselected_df[x_col],
                y=unselected_df[y_col],
                mode='markers',
                marker=dict(color='lightgray', size=point_size, opacity=0.3),
                customdata=unselected_df[['id']],
                showlegend=False,
                hoverinfo='skip'
            )
        )

    # Selected points, grouped by `group`
    if len(selected_df) > 0:
        assert group in selected_df.columns, f"Grouping column '{group}' not found in data."
        hover_columns = [group] + [c for c in visible_columns if c != group] if not minimal_hover else ['id']
        hovertemplate = get_hover_text(hover_columns)
        
        for group_name, group_df in selected_df.groupby(group, sort=False):
            fig.add_trace(
                go.Scattergl(
                    x=group_df[x_col],
                    y=group_df[y_col],
                    mode='markers',
                    marker=dict(size=point_size, color=color_map.get(group_name, 'blue')),
                    name=str(group_name),
                    #customdata=group_df[hover_columns].values,
                    customdata=group_df[hover_columns].values if not minimal_hover else group_df[['id']].values,
                    hovertemplate=hovertemplate
                )
            )

    
    fig.update_layout(
        template='plotly_white',
        margin=dict(l=0, r=0, t=20, b=0),
        showlegend=True,
        xaxis_title=x_col,
        yaxis_title=y_col
    )

    return fig


##----------------------------------------------------
## map
def get_map_plot(selected_ids, group, point_size, color_map, visible_columns, minimal_hover=False):
    selected_df = get_selected_df(selected_ids)
    unselected_df = get_unselected_df(selected_ids)

    traces = []

    # Base map with unselected (gray)
    if not unselected_df.empty:
        traces.append(
            go.Scattermap(
                lat=unselected_df[ANNOTATION_LAT],
                lon=unselected_df[ANNOTATION_LONG],
                mode='markers',
                marker=dict(size=point_size, color='lightgray', opacity=0.3),
                customdata=unselected_df[["id"]].to_numpy(),
                showlegend=False,
                hoverinfo='skip'  # Optional: speed up by disabling hover
            )
        )


    # Add selected trace
    if not selected_df.empty:
        assert group in selected_df.columns, f"Grouping column '{group}' not found in data."
        hover_columns = [group] + [c for c in visible_columns if c != group] if not minimal_hover else ['id']
        hovertemplate = get_hover_text(hover_columns)

        # ✅ Single trace instead of per-group loop
        traces.append(
            go.Scattermap(
                lat=selected_df[ANNOTATION_LAT],
                lon=selected_df[ANNOTATION_LONG],
                mode="markers",
                marker=dict(
                    size=point_size,
                    color=selected_df[group].map(lambda g: color_map.get(g, "blue")),
                ),
                customdata=selected_df[hover_columns].to_numpy(),
                hovertemplate=hovertemplate,
                showlegend=True,
            )
        )

        '''
        # ❌ Old code: per-group traces (inefficient)
        for g_val, sub_df in selected_df.groupby(group):
            traces.append(
                go.Scattermap(
                    lat=sub_df[ANNOTATION_LAT],
                    lon=sub_df[ANNOTATION_LONG],
                    mode='markers',
                    marker=dict(size=point_size, color=color_map.get(g_val, 'blue')),
                    customdata=sub_df[hover_columns].values,
                    hovertemplate=hovertemplate,
                    showlegend=False
                )
            )
        '''

    # Create figure once
    fig = go.Figure(data=traces)
    fig.update_layout(
        mapbox_style='open-street-map',
        margin={'l': 0, 'r': 0, 't': 0, 'b': 0},
        showlegend=False
    )

    return fig


##----------------------------------------------------
## time histogram
def get_time_histogram(selected_ids):
    selected_df = get_selected_df(selected_ids)

    # All points
    all_hist = px.histogram(
        df,
        x=ANNOTATION_TIME,
        nbins=args.time_hist_nbins,
        opacity=0.3,
        color_discrete_sequence=['lightgray'],
        labels={ANNOTATION_TIME: 'Time'}
    )

    # Selected points
    if len(selected_df) > 0:
        selected_hist = px.histogram(
            selected_df,
            x=ANNOTATION_TIME,
            nbins=args.time_hist_nbins,
            opacity=0.9,
            color_discrete_sequence=['#1F77B4'],
            labels={ANNOTATION_TIME: 'Time'}
        )

        fig = go.Figure(data=all_hist.data + selected_hist.data)

        start = int(selected_df[ANNOTATION_TIME].min())
        end = int(selected_df[ANNOTATION_TIME].max())

    else:
        fig = go.Figure(data=all_hist.data)

        start = int(df[ANNOTATION_TIME].min())
        end = int(df[ANNOTATION_TIME].max())


    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        title="",
        barmode='overlay',  
        xaxis=dict(
            title=f"Time range from {start:,} to {end:,}",
            rangeslider=dict(visible=True),
            type="linear"
        ),
        uirevision='time-plot'
    )

    return fig

## time scatter
def get_time_scatter(selected_ids, group, point_size, color_map, visible_columns):
    logging.debug(f"   get_time_scatter() ... ")

    selected_df = get_selected_df(selected_ids)
    unselected_df = get_unselected_df(selected_ids)

    # Jitter for y-axis
    def jitter(n, scale=1.0):
        return np.random.uniform(-scale, scale, n)

    fig = go.Figure()

    # Unselected points (gray)
    if len(unselected_df) > 0:
        fig.add_trace(
            go.Scattergl(
                x=unselected_df[ANNOTATION_TIME],
                y=jitter(len(unselected_df), scale=0.5),
                mode='markers',
                marker=dict(color='lightgray', size=point_size, opacity=0.3),
                customdata=unselected_df[['id']],
                showlegend=False,
                hoverinfo='skip'  # Optional: speed up by disabling hover
            )
        )

    # Selected points, grouped by `group`
    if len(selected_df) > 0:
        assert group in selected_df.columns, f"Grouping column '{group}' not found in data."
        hover_columns = [group] + [c for c in (visible_columns or []) if c != group] if not minimal_hover else ['id']
        hovertemplate = get_hover_text(hover_columns)

        for group_name, group_df in selected_df.groupby(group, sort=False):
            fig.add_trace(
                go.Scattergl(
                    x=group_df[ANNOTATION_TIME],
                    y=jitter(len(group_df), scale=0.5),
                    mode='markers',
                    marker=dict(size=point_size, color=color_map.get(group_name, 'blue')),
                    name=str(group_name),
                    customdata=group_df[hover_columns].values,
                    hovertemplate=hovertemplate
                )
            )

        start = int(selected_df[ANNOTATION_TIME].min())
        end = int(selected_df[ANNOTATION_TIME].max())

    else:
        start = int(df[ANNOTATION_TIME].min())
        end = int(df[ANNOTATION_TIME].max())

    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        title="",
        xaxis=dict(
            title=f"Time range from {start:,} to {end:,}",
            rangeslider=dict(visible=True),
            type="linear"
        ),
        yaxis=dict(
            title="Jitter",
            showticklabels=False,
            zeroline=False
        ),
        uirevision='time-plot'
    )
    logging.debug(f"   get_time_scatter() ... done.")
    return fig



def make_scatter(x, y, selected_ids, group, point_size, color_map, visible_groups, xaxis_title, yaxis_title):
    selected = df[df["id"].isin(selected_ids)]
    unselected = df[~df["id"].isin(selected_ids)]

    # Jitter for y-axis
    def jitter(n, scale=1.0):
        return np.random.uniform(-scale, scale, n)
    
    traces = []
    if not unselected.empty:
        y = unselected[y] if y is not None else jitter(len(unselected), scale=0.5)
        traces.append(go.Scatter(
            x=unselected[x], 
            y=y,
            mode="markers",
            marker=dict(size=point_size, color="lightgray", opacity=0.2),
            customdata=unselected[["id"]],
            hoverinfo="skip",
            showlegend=False
        ))

    if not selected.empty:
        for g, sub in selected.groupby(group):
            if g not in visible_groups:
                continue
            y = selected[y] if y is not None else jitter(len(selected), scale=0.5)
            traces.append(go.Scatter(
                x=sub[x], 
                y=y,
                mode="markers",
                marker=dict(size=point_size, color=color_map.get(g, "black")),
                customdata=sub[["id"]],
                name=g
            ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        template='plotly_white',
        margin=dict(l=20, r=20, t=20, b=20),
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle"),
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title
    )
    return fig


def get_pca_plot(selected_ids, color_map, visible_groups, x_col, y_col, group, point_size):
    logging.debug(f"   get_pca_plot({len(selected_ids)} selected, {len(color_map)} nb color_map, {len(visible_groups)} nb visible_groups, {group} group, {x_col} x, {y_col} y")
    return make_scatter(x_col, y_col, selected_ids, group, point_size, color_map, visible_groups, "PCA", "C1")


def get_map_plot(selected_ids, color_map, visible_groups, group, point_size):
    logging.debug(f"   get_map_plot({len(selected_ids)} selected, {len(color_map)} nb color_map, {len(visible_groups)} nb visible_groups, {group} group")
    selected = df[df["id"].isin(selected_ids)]
    unselected = df[~df["id"].isin(selected_ids)]

    traces = []
    if not unselected.empty:
        traces.append(go.Scattergeo(
            lat=unselected[ANNOTATION_LAT], lon=unselected[ANNOTATION_LONG],
            mode="markers",
            marker=dict(size=point_size, color="lightgray", opacity=0.2),
            customdata=unselected[["id"]],
            hoverinfo="skip",
            showlegend=False
        ))

    if not selected.empty:
        for g, sub in selected.groupby(group):
            if g not in visible_groups:
                continue
            traces.append(go.Scattergeo(
                lat=sub[ANNOTATION_LAT], lon=sub[ANNOTATION_LONG],
                mode="markers",
                marker=dict(size=point_size, color=color_map.get(g, "black")),
                customdata=sub[["id"]],
                name=g
            ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        geo=dict(projection_type="equirectangular"),
        margin=dict(l=20, r=20, t=20, b=20),
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle")
    )
    return fig


def get_time_plot(selected_ids, color_map, visible_groups, group, point_size):
    logging.debug(f"   get_time_plot({len(selected_ids)} selected, {len(color_map)} nb color_map, {len(visible_groups)} nb visible_groups, {group} group")
    return make_scatter(ANNOTATION_TIME, None, selected_ids, group, point_size, color_map, visible_groups, "Time", "")


##----------------------------------------------------
## time logger
def log_time(name=None):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            try:                
                trigger = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
            except Exception as e:
                trigger = "no trigger"
            result = func(*args, **kwargs)
            elapsed = time.perf_counter() - start
            logging.debug(f"{elapsed:.3f}s used by callback {name or func.__name__}({trigger}).")
            return result
        return wrapper
    return decorator
## -------------------------------------------------------------------
### callbacks

## dcc.Store color map update callback
@app.callback(
    Output('color-map', 'data'),
    Input('group-selector', 'value'),
)
@log_time("update_color_map")
def update_color_map(group):
    assert group in df.columns, f"Group column '{group}' not found in dataframe columns."

    unique_values = sorted(df[group].dropna().unique())
    assert len(unique_values) > 0, f"No unique values found in group column '{group}'."

    px_colors = px.colors.qualitative.Plotly
    color_map = {val: px_colors[i % len(px_colors)] for i, val in enumerate(unique_values)}
    return color_map




    if args.time_hist:
        triggered_id = [t['prop_id'].split('.')[0] for t in callback_context.triggered][0] if callback_context.triggered else None
        if triggered_id != 'selected-ids':
            return dash.no_update
        return get_time_histogram(selected_indices)
    else:
        return get_time_scatter(selected_indices, group, point_size, color_map, visible_columns)
    

### ---------------------------------------------------
## Selected count message update callback (independent, can be called safely)
@app.callback(
    Output("selection-count", "children"),
    Input('selected-ids', 'data'),
)
@log_time("update_selection_count")
def update_selection_count(selected_indices):
    total = len(df)
    selected = len(selected_indices)
    return f"{selected} of {total} points selected"


### ---------------------------------------------------
# Column selection callback
@app.callback(
    Output('visible-columns', 'data', allow_duplicate=False),
    Input('annotation-table', 'selectedRows'),
    prevent_initial_call=True
)
@log_time("update_visible_columns")
def update_visible_columns(selected_rows):
    if not selected_rows:
        return ['id']
    # Use set for uniqueness, keep order
    selected_columns = ['id'] + [row['Abbreviation'] for row in selected_rows if row['Abbreviation'] != 'id']
    return selected_columns


### ---------------------------------------------------
# Data table columns update
@app.callback(
    Output('data-table', 'columnDefs', allow_duplicate=False),
    Input('visible-columns', 'data'),
    prevent_initial_call=True
)
@log_time("update_data_table_columns")
def update_data_table_columns(selected_columns):
    if not selected_columns:
        return dash.no_update
    defs = [{'headerName': col, 'field': col, 'sortable': True, 'resizable': True,
            **({'checkboxSelection': True, 'headerCheckboxSelection': True} if col == 'id' else {})}
            for col in selected_columns]
    return defs


### ---------------------------------------------------
### ---------------------------------------------------
## selection changes
@app.callback(
    Output('selected-ids', 'data'),
    Input('texarea-expression', 'value'),
    Input('time-plot', 'relayoutData'),
    Input('data-table', 'selectedRows'),
    Input('pca-plot', 'selectedData'),
    Input('pca-plot', 'restyleData'), 
    Input('map-plot', 'selectedData'),
    prevent_initial_call=True
)
@log_time("update_selected_indices")
def update_selected_indices(texarea_query, time_selection, table_selection, pca_selection, pca_restyle, map_selection):
    trigger_name = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None

    if trigger_name is None:
        sel = dash.no_update

    ## texarea-expression
    elif trigger_name == "texarea-expression":
        if annotation is None:
            return dash.no_update

        # empty input -> select all
        if not texarea_query or str(texarea_query).strip() == "":
            sel = df['id'].tolist()
        try:
            # Use pandas eval for simple expressions, fallback to query for complex ones
            # This is faster for simple filters
            if "==" in texarea_query or ">" in texarea_query or "<" in texarea_query:
                filtered_df = df.eval(texarea_query)
                sel = df[filtered_df]['id'].tolist()
            else:
                filtered_df = df.query(texarea_query)
                sel = filtered_df['id'].tolist()
        except Exception as e:
            print(f"Query failed: {e}")
            sel = df['id'].tolist()

    ## time-plot
    elif trigger_name == "time-plot":
        if ANNOTATION_TIME is None:
            return dash.no_update
        if not time_selection:
            sel = df['id'].tolist()
        x_range = time_selection.get('xaxis.range', None)
        if x_range is None:
            x0 = time_selection.get('xaxis.range[0]')
            x1 = time_selection.get('xaxis.range[1]')
            if x0 is not None and x1 is not None:
                x_range = [x0, x1]
        if x_range:
            filtered_df = df[(df[ANNOTATION_TIME] >= x_range[0]) & (df[ANNOTATION_TIME] <= x_range[1])]
            sel = filtered_df['id'].tolist()
        sel = df['id'].tolist()
    
    ## data-table
    elif trigger_name == "data-table":
        if not table_selection:
            sel = []
        selected_ids = [row['id'] for row in table_selection if 'id' in row]
        sel = selected_ids

    elif trigger_name == "pca-plot" and pca_restyle is not None:
        # pca_restyle is a list like [{'visible': [True, False, ...]}, [trace_indices]]
        # We want to select all points in visible traces (i.e., visible groups)
        if isinstance(pca_restyle, list) and len(pca_restyle) == 2:
            style_dict, trace_indices = pca_restyle
            if 'visible' in style_dict:
                # Get the current group column
                group_col = ctx.inputs_list[3]['value']['group'] if 'group' in ctx.inputs_list[3]['value'] else None
                if group_col and group_col in df.columns:
                    # Get all group values in the same order as traces
                    group_values = df[group_col].dropna().unique().tolist()
                    # Build a mask for visible groups
                    visible_flags = style_dict['visible']
                    # trace_indices tells which traces were affected, but we want all visible
                    # So, get all currently visible groups
                    # If visible_flags is a list of bools for all traces:
                    visible_groups = [g for g, v in zip(group_values, visible_flags) if v]
                    # Select all points in visible groups
                    visible_ids = df[df[group_col].isin(visible_groups)]['id'].tolist()
                    sel = visible_ids
        # Fallback: select all points
        sel = df['id'].tolist()
    
    ## pca-plot
    elif trigger_name == "pca-plot":
        if not pca_selection or 'points' not in pca_selection:
            return dash.no_update
        sel = [point['customdata'] for point in pca_selection['points'] if 'customdata' in point]


    ## map-plot
    elif trigger_name == "map-plot":
        if not map_selection or 'points' not in map_selection:
            return dash.no_update
        sel = [point['customdata'] for point in map_selection['points'] if 'customdata' in point]
    

    return sel if sel is not None else dash.no_update


@app.callback(
    Output("visible-groups", "data"),
    Input("pca-plot", "restyleData"),
    Input("map-plot", "restyleData"),
    Input("time-plot", "restyleData"),
    State("visible-groups", "data"),
    State("color-map", "data"),
    prevent_initial_call=False
)
@log_time("update_visible_groups")
def update_visible_groups(pca_style, geo_style, time_style, current, color_map):
    ctx = dash.callback_context.triggered_id
    print(f"update_visible_groups triggered by {ctx}")
    
    if ctx == "pca-plot":
        style = pca_style
    elif ctx == "map-plot":
        style = geo_style
    elif ctx == "time-plot":
        style = time_style
    else:
        style = None

    if style and "visible" in style[0]:
        visibility, traces = style
        traces = set(traces)
        if "legendonly" in visibility.values():
            new = [g for i, g in enumerate(current) if i not in traces]
        else:
            new = list(set(current) | set(color_map))
        return new
    return current


@app.callback(
    Output("pca-plot", "figure"),
    Output("map-plot", "figure"),
    Output("time-plot", "figure"),
    Input("selected-ids", "data"),
    Input("visible-groups", "data"),
    Input("color-map", "data"),
    State('x-axis-selector', 'value'),
    State('y-axis-selector', 'value'),
    State('group-selector', 'value'),
    State('point-size-selector', 'value'),

)
def update_plots(selected_ids, visible_groups, color_map, x_col, y_col, group, point_size):
    return (
        get_pca_plot(selected_ids, color_map, visible_groups, x_col, y_col, group, point_size),
        get_map_plot(selected_ids, color_map, visible_groups, group, point_size),
        get_time_plot(selected_ids, color_map, visible_groups, group, point_size)
    )

### ---------------------------------------------------
### ---------------------------------------------------
## Launch server
if __name__ == '__main__':
    logging.info(f"Dashboard running at http://localhost:{args.server_port}")
    app.run(debug=True, port=args.server_port)


2025-09-14 21:05:52,701 [DEBUG] get_pca_tab() ... 
2025-09-14 21:05:52,702 [DEBUG]    get_selectors() ... 
2025-09-14 21:05:52,704 [DEBUG]    get_selectors() ... done.
2025-09-14 21:05:52,705 [DEBUG]    get_data_table() ... 
2025-09-14 21:05:52,999 [DEBUG]    get_data_table() ... done.
2025-09-14 21:05:53,001 [DEBUG] get_pca_tab() ... done.
2025-09-14 21:05:53,004 [DEBUG] get_annotation_tab() ... 
2025-09-14 21:05:53,008 [DEBUG] get_annotation_tab() ... done.
2025-09-14 21:05:53,011 [DEBUG] get_eigenvalues_tab() ... 
2025-09-14 21:05:53,015 [DEBUG] get_eigenvalues_tab() ... done
2025-09-14 21:05:53,021 [DEBUG] get_statistics_tab() ... 
2025-09-14 21:05:53,120 [DEBUG] get_statistics_tab() ... done.
2025-09-14 21:05:53,122 [DEBUG] get_help_tab() ... 
2025-09-14 21:05:53,125 [DEBUG] get_help_tab() ... done.
2025-09-14 21:05:53,208 [INFO] Dashboard running at http://localhost:8050
2025-09-14 21:05:53,719 [DEBUG] Starting new HTTP connection (1): 127.0.0.1:8050
2025-09-14 21:05:53,721 [DEBU

2025-09-14 21:05:55,660 [DEBUG] 0.000s used by callback update_visible_groups(None).
2025-09-14 21:05:55,663 [DEBUG] 0.001s used by callback update_color_map(None).
2025-09-14 21:05:55,667 [DEBUG] 0.000s used by callback update_selection_count(None).
2025-09-14 21:05:55,684 [DEBUG]    get_pca_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group, C1 x, C2 y
2025-09-14 21:05:55,711 [DEBUG]    get_map_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:05:55,726 [DEBUG]    get_time_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group


update_visible_groups triggered by None


2025-09-14 21:06:39,095 [DEBUG] 0.000s used by callback update_selected_indices(time-plot).
2025-09-14 21:06:39,125 [DEBUG] 0.000s used by callback update_selection_count(selected-ids).
2025-09-14 21:06:39,128 [DEBUG]    get_pca_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group, C1 x, C2 y
2025-09-14 21:06:39,193 [DEBUG]    get_map_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:06:39,242 [DEBUG]    get_time_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:07:14,088 [DEBUG] 0.000s used by callback update_selection_count(None).
2025-09-14 21:07:14,130 [DEBUG] 0.002s used by callback update_color_map(None).
2025-09-14 21:07:14,137 [DEBUG] 0.001s used by callback update_visible_groups(None).


update_visible_groups triggered by None


2025-09-14 21:07:17,687 [DEBUG]    get_pca_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group, C1 x, C2 y
2025-09-14 21:07:17,714 [DEBUG]    get_map_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:07:17,729 [DEBUG]    get_time_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:16:53,989 [DEBUG] 0.001s used by callback update_selected_indices(time-plot).
2025-09-14 21:17:09,903 [DEBUG]    get_pca_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group, C1 x, C2 y
2025-09-14 21:17:09,911 [DEBUG] 0.000s used by callback update_selection_count(selected-ids).
2025-09-14 21:17:09,947 [DEBUG]    get_map_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
2025-09-14 21:17:09,963 [DEBUG]    get_time_plot(10442 selected, 173 nb color_map, 0 nb visible_groups, Skeletal_elemen group
